In [20]:
# ===========================
#  Merge the data
# ===========================
import pandas as pd
from pathlib import Path

OUT_DIR   = Path("outputs")
ID_COL    = "essay_id_comp"
GROUP_COL = "gender"
TARGET_COL= "holistic_essay_score_3cat"

# Load test set ground truth
test_df = pd.read_csv(OUT_DIR / "test_split.csv", usecols=[ID_COL, GROUP_COL, TARGET_COL])

# Prediction files (already saved in outputs/)
pred_paths = {
    "tfidf":            OUT_DIR / "scores_tfidf_ordlogit_cv.csv",
    "bert":             OUT_DIR / "scores_bert_ord.csv",
    "chatgpt_zeroshot": OUT_DIR / "scores_chatgpt_zeroshot.csv",
    "chatgpt_fewshot":  OUT_DIR / "scores_chatgpt_fewshot.csv",
}

dfs = []
for model_key, path in pred_paths.items():
    df = pd.read_csv(path)
    pred_col = [c for c in df.columns if c.startswith("y_pred")][0]
    df = df[[ID_COL, pred_col]].rename(columns={pred_col: f"y_pred_{model_key}"})
    dfs.append(df)

# Merge all predictions onto the test set
merged = test_df.copy()
for mdf in dfs:
    merged = merged.merge(mdf, on=ID_COL, how="left")

merged_path = OUT_DIR / "scores_merged_test.csv"
merged.to_csv(merged_path, index=False)

print(">>> Merge complete (TEST only)")
print("Shape:", merged.shape)
print("Columns:", list(merged.columns))

>>> Merge complete (TEST only)
Shape: (434, 7)
Columns: ['essay_id_comp', 'gender', 'holistic_essay_score_3cat', 'y_pred_tfidf', 'y_pred_bert', 'y_pred_chatgpt_zeroshot', 'y_pred_chatgpt_fewshot']


In [19]:
merged.head()

,essay_id_comp,gender,holistic_essay_score,y_pred_tfidf,y_pred_bert,y_pred_chatgpt_zeroshot,y_pred_chatgpt_fewshot
0,7F220FD6CACF,M,2,1,2,1,1
1,B3FE48815543,F,6,3,2,2,2
2,B6428F6EB5F3,F,3,2,2,2,2
3,5D4E78CE6D5F,F,4,2,2,1,2
4,EA48C80121AD,F,3,1,1,1,1


In [21]:
# ===========================
#  Block 1 — Agreement (Kappa & QWK)
# ===========================
from sklearn.metrics import cohen_kappa_score
import pandas as pd

MODEL_COLS = {
    "tfidf": "y_pred_tfidf",
    "bert": "y_pred_bert",
    "chatgpt_zeroshot": "y_pred_chatgpt_zeroshot",
    "chatgpt_fewshot": "y_pred_chatgpt_fewshot",
}

def _pair_clean(y_true, y_pred):
    s = pd.concat([y_true, y_pred], axis=1).dropna()
    return s.iloc[:,0].astype(int), s.iloc[:,1].astype(int)

rows = []
for name, col in MODEL_COLS.items():
    yt, yp = _pair_clean(merged[TARGET_COL], merged[col])
    kappa_plain = cohen_kappa_score(yt, yp)
    kappa_quadr = cohen_kappa_score(yt, yp, weights="quadratic")
    rows.append([name, kappa_plain, kappa_quadr])

block1_df = pd.DataFrame(rows, columns=["model", "kappa", "qwk"]).set_index("model")
print("\n=== Block 1 — Agreement (vs. holistic_essay_score) ===")
print(block1_df.round(3))



=== Block 1 — Agreement (vs. holistic_essay_score) ===
                  kappa    qwk
model                         
tfidf             0.612  0.710
bert              0.712  0.792
chatgpt_zeroshot  0.240  0.369
chatgpt_fewshot   0.401  0.516


In [23]:
# ===========================
#  Block 2 — Fairness: DI (Uncond., Cond@1..3)
# ===========================
import numpy as np
import pandas as pd

# (Using your MODEL_COLS defined earlier in Block 1)
# If you need a fallback, uncomment below:
# MODEL_COLS = {
#     "tfidf": "y_pred_tfidf",
#     "bert": "y_pred_bert",
#     "chatgpt_zeroshot": "y_pred_chatgpt_zeroshot",
#     "chatgpt_fewshot": "y_pred_chatgpt_fewshot",
# }

# True band is already 1–3 in TARGET_COL
true_band = merged[TARGET_COL].astype("Int64")

def di_on_event(df_idx, event_mask):
    """
    Disparate Impact (DI) on event A:
      DI = P(A | G=F) / P(A | G=M)
    No EPS; if a denominator is zero or missing, returns NaN.
    """
    sub = merged.loc[df_idx]
    isF = (sub[GROUP_COL] == "F")
    isM = (sub[GROUP_COL] == "M")

    # Align event mask to subset indices
    if isF.any():
        p_f = event_mask.loc[isF[isF].index].mean()
    else:
        p_f = np.nan

    if isM.any():
        p_m = event_mask.loc[isM[isM].index].mean()
    else:
        p_m = np.nan

    # # Optional stabilization (disabled as requested)
    # eps = 1e-9
    # return (p_f + eps) / (p_m + eps)

    # No EPS: guard divide-by-zero/NaN
    if pd.isna(p_f) or pd.isna(p_m) or p_m == 0:
        return np.nan
    return p_f / p_m

def di_table_for_model(pred_col):
    # Predictions already in 1–3
    pred_band = merged[pred_col].astype("Int64")

    # Events A_k := (pred_band == k)
    A = {k: (pred_band == k) for k in (1, 2, 3)}

    # Unconditional DI: geometric mean of DI_k over k in {1,2,3}
    di_k = [di_on_event(merged.index, A[k]) for k in (1, 2, 3)]
    arr = np.array(di_k, dtype=float)
    if np.any(arr <= 0) or np.all(np.isnan(arr)):
        uncond = np.nan
    else:
        uncond = float(np.exp(np.nanmean(np.log(arr))))

    # Conditional DI at true band k
    out = {"Uncond.": uncond}
    for k in (1, 2, 3):
        idx_k = merged.index[true_band == k]
        out[f"Cond@{k}"] = di_on_event(idx_k, A[k])
    return out

rows = []
for model_name, col in MODEL_COLS.items():
    res = di_table_for_model(col)
    rows.append([model_name, res["Uncond."], res["Cond@1"], res["Cond@2"], res["Cond@3"]])

block2_di_df = pd.DataFrame(rows, columns=["model", "Uncond.", "Cond@1", "Cond@2", "Cond@3"]).set_index("model")
print("\n=== Block 2 — Fairness (DI) on 3-category outcome ===")
print(block2_di_df.round(3))



=== Block 2 — Fairness (DI) on 3-category outcome ===
                  Uncond.  Cond@1  Cond@2  Cond@3
model                                            
tfidf               1.035   0.988   1.005   0.758
bert                1.059   1.008   0.946   0.885
chatgpt_zeroshot    1.096   0.936   1.207     NaN
chatgpt_fewshot     1.022   0.872   1.050   0.833


In [27]:
# ===========================
#  Block 2 — Fairness: DI (Uncond., Cond@1..3) — normalized Summary
# ===========================
import numpy as np
import pandas as pd

# Using your existing globals: merged, GROUP_COL, TARGET_COL, MODEL_COLS
true_band = merged[TARGET_COL].astype("Int64")

def _di_for_subset(pred_series, sub_idx, r):
    """
    DI for event A_r: (pred == r) within subset sub_idx.
    DI = P(A_r|F) / P(A_r|M).
    No EPS; returns NaN if undefined (e.g., no M or p_m == 0).
    """
    idx = merged.index.intersection(pd.Index(sub_idx))
    if len(idx) == 0:
        return np.nan

    idx_f = idx[merged.loc[idx, GROUP_COL] == "F"]
    idx_m = idx[merged.loc[idx, GROUP_COL] == "M"]
    n_f, n_m = len(idx_f), len(idx_m)
    if n_f == 0 or n_m == 0:
        return np.nan

    num_f = int((pred_series.loc[idx_f] == r).sum())
    num_m = int((pred_series.loc[idx_m] == r).sum())
    p_f = num_f / n_f
    p_m = num_m / n_m
    if p_m == 0:
        return np.nan
    return p_f / p_m

def _weights_for_subset(pred_series, sub_idx):
    """
    Raw class-mix weights w_r(S) = #(pred==r in S) / |S| for r=1,2,3.
    Returns dict {1: w1, 2: w2, 3: w3}. If |S|=0 -> all NaN.
    """
    idx = merged.index.intersection(pd.Index(sub_idx))
    n = len(idx)
    if n == 0:
        return {1: np.nan, 2: np.nan, 3: np.nan}
    w = {}
    for r in (1, 2, 3):
        w[r] = float((pred_series.loc[idx] == r).sum()) / n
    return w

def di_table_for_model(pred_col):
    """
    Build a 4x4 table for one model:
      Rows: Pred=1, Pred=2, Pred=3, Summary
      Cols: Uncond., Cond@1, Cond@2, Cond@3
    Summary uses normalized weights from the subset’s prediction mix.
    """
    pred = merged[pred_col].astype("Int64")

    subsets = {
        "Uncond.": merged.index,
        "Cond@1": merged.index[true_band == 1],
        "Cond@2": merged.index[true_band == 2],
        "Cond@3": merged.index[true_band == 3],
    }

    # Compute DI for each predicted class r and each subset column
    rows = []
    di_grid = {r: {} for r in (1, 2, 3)}
    for r in (1, 2, 3):
        for col_name, sub_idx in subsets.items():
            di_grid[r][col_name] = _di_for_subset(pred, sub_idx, r)

    # Build the first three rows (Pred=1..3)
    for r in (1, 2, 3):
        rows.append([
            f"Pred={r}",
            di_grid[r]["Uncond."],
            di_grid[r]["Cond@1"],
            di_grid[r]["Cond@2"],
            di_grid[r]["Cond@3"],
        ])

    # Summary row (normalized weights within each column subset)
    summary_vals = []
    for col_name, sub_idx in subsets.items():
        w_raw = _weights_for_subset(pred, sub_idx)  # {1:w1,2:w2,3:w3}
        # print(f"Debug: weights for {col_name}: {w_raw}")
        # keep only classes with defined DI
        valid = [r for r in (1, 2, 3) if (not pd.isna(di_grid[r][col_name]))]
        if len(valid) == 0:
            summary_vals.append(np.nan)
            continue
        denom = sum(w_raw[r] for r in valid if not pd.isna(w_raw[r]))
        if denom == 0 or pd.isna(denom):
            summary_vals.append(np.nan)
            continue
        val = sum((w_raw[r] / denom) * di_grid[r][col_name] for r in valid)
        summary_vals.append(val)

    rows.append(["Summary"] + summary_vals)

    out = pd.DataFrame(
        rows,
        columns=["Row", "Uncond.", "Cond@1", "Cond@2", "Cond@3"]
    ).set_index("Row")

    return out

# Run for each method and print
for model_name, col in MODEL_COLS.items():
    table = di_table_for_model(col)
    print(f"\n=== Block 2 — DI (normalized summary) — {model_name} ===")
    print(table.round(3))



=== Block 2 — DI (normalized summary) — tfidf ===
         Uncond.  Cond@1  Cond@2  Cond@3
Row                                     
Pred=1     0.778   0.988   0.792     NaN
Pred=2     1.134   1.049   1.005   1.296
Pred=3     1.254     NaN   1.706   0.758
Summary    1.016   1.000   1.012   1.039

=== Block 2 — DI (normalized summary) — bert ===
         Uncond.  Cond@1  Cond@2  Cond@3
Row                                     
Pred=1     0.823   1.008   1.253     NaN
Pred=2     1.054   0.969   0.946   1.458
Pred=3     1.370     NaN   1.754   0.885
Summary    1.012   1.000   1.016   1.029

=== Block 2 — DI (normalized summary) — chatgpt_zeroshot ===
         Uncond.  Cond@1  Cond@2  Cond@3
Row                                     
Pred=1     0.848   0.936   0.881   0.833
Pred=2     1.415   2.906   1.207   1.042
Pred=3       NaN     NaN     NaN     NaN
Summary    1.030   1.049   1.012   1.004

=== Block 2 — DI (normalized summary) — chatgpt_fewshot ===
         Uncond.  Cond@1  Cond@2  Cond

In [30]:
# ===========================
#  Block 2 — Fairness: DP (Uncond., Cond@1..3) — normalized Summary
# ===========================
import numpy as np
import pandas as pd

true_band = merged[TARGET_COL].astype("Int64")

def _dp_for_subset(pred_series, sub_idx, r):
    """
    DP_r(S) = P(pred=r | F, S) - P(pred=r | M, S).
    Returns NaN only if a group is absent in S; zeros are valid results.
    """
    idx = merged.index.intersection(pd.Index(sub_idx))
    if len(idx) == 0:
        return np.nan

    idx_f = idx[merged.loc[idx, GROUP_COL] == "F"]
    idx_m = idx[merged.loc[idx, GROUP_COL] == "M"]
    n_f, n_m = len(idx_f), len(idx_m)
    if n_f == 0 or n_m == 0:
        return np.nan

    num_f = int((pred_series.loc[idx_f] == r).sum())
    num_m = int((pred_series.loc[idx_m] == r).sum())
    p_f = num_f / n_f
    p_m = num_m / n_m
    return p_f - p_m  # may be zero; keep it

def _pred_weights(pred_series, sub_idx):
    """w_r(S) = #(pred==r in S) / |S|; zeros allowed; NaN only if S empty."""
    idx = merged.index.intersection(pd.Index(sub_idx))
    n = len(idx)
    if n == 0:
        return {1: np.nan, 2: np.nan, 3: np.nan}
    return {r: float((pred_series.loc[idx] == r).sum()) / n for r in (1,2,3)}

def dp_table_for_model(pred_col):
    pred = merged[pred_col].astype("Int64")
    subsets = {
        "Uncond.": merged.index,
        "Cond@1": merged.index[true_band == 1],
        "Cond@2": merged.index[true_band == 2],
        "Cond@3": merged.index[true_band == 3],
    }

    # per-row values
    dp_grid = {r: {} for r in (1,2,3)}
    for r in (1,2,3):
        for col_name, sub_idx in subsets.items():
            dp_grid[r][col_name] = _dp_for_subset(pred, sub_idx, r)

    # table rows (Pred=1..3)
    rows = []
    for r in (1,2,3):
        rows.append([
            f"Pred={r}",
            dp_grid[r]["Uncond."],
            dp_grid[r]["Cond@1"],
            dp_grid[r]["Cond@2"],
            dp_grid[r]["Cond@3"],
        ])

    # Summary (prediction-mix weights within each subset; zeros included; NaNs excluded via renorm)
    summary_vals = []
    for col_name, sub_idx in subsets.items():
        w_raw = _pred_weights(pred, sub_idx)
        print(f"Debug: weights for {col_name}: {w_raw}")
        valid_r = [r for r in (1,2,3) if not pd.isna(dp_grid[r][col_name])]
        if len(valid_r) == 0:
            summary_vals.append(np.nan)
            continue
        denom = sum(w_raw[r] for r in valid_r if not pd.isna(w_raw[r]))
        if denom is None or pd.isna(denom) or denom == 0:
            summary_vals.append(np.nan)
            continue
        val = sum((w_raw[r] / denom) * dp_grid[r][col_name] for r in valid_r)
        summary_vals.append(val)

    rows.append(["Summary"] + summary_vals)

    return (pd.DataFrame(rows, columns=["Row","Uncond.","Cond@1","Cond@2","Cond@3"])
              .set_index("Row"))

# Run for each method
for model_name, col in MODEL_COLS.items():
    table = dp_table_for_model(col)
    print(f"\n=== Block 2 — DP (normalized summary) — {model_name} ===")
    print(table.round(3))


Debug: weights for Uncond.: {1: 0.35714285714285715, 2: 0.5691244239631337, 3: 0.07373271889400922}
Debug: weights for Cond@1: {1: 0.802547770700637, 2: 0.19745222929936307, 3: 0.0}
Debug: weights for Cond@2: {1: 0.12446351931330472, 2: 0.8283261802575107, 3: 0.04721030042918455}
Debug: weights for Cond@3: {1: 0.0, 2: 0.5227272727272727, 3: 0.4772727272727273}

=== Block 2 — DP (normalized summary) — tfidf ===
         Uncond.  Cond@1  Cond@2  Cond@3
Row                                     
Pred=1    -0.088  -0.010  -0.029   0.000
Pred=2     0.072   0.010   0.004   0.133
Pred=3     0.017   0.000   0.025  -0.133
Summary    0.010  -0.006   0.001   0.006
Debug: weights for Uncond.: {1: 0.3294930875576037, 2: 0.5622119815668203, 3: 0.10829493087557604}
Debug: weights for Cond@1: {1: 0.8089171974522293, 2: 0.1910828025477707, 3: 0.0}
Debug: weights for Cond@2: {1: 0.06866952789699571, 2: 0.871244635193133, 3: 0.060085836909871244}
Debug: weights for Cond@3: {1: 0.0, 2: 0.25, 3: 0.75}

=== B

In [31]:
# ===========================
#  Block 2 — Fairness: EO (Uncond., Cond@1..3) — normalized Summary
# ===========================
import numpy as np
import pandas as pd

true_band = merged[TARGET_COL].astype("Int64")

def _eo_cond_k_for_pred_r(pred_series, k, r):
    """
    EO_r(k) = P(pred=r | true=k, F) - P(pred=r | true=k, M).
    Returns NaN only if a group is absent in S_k; zeros are valid.
    """
    S_k = merged.index[true_band == k]
    idx = merged.index.intersection(S_k)
    if len(idx) == 0:
        return np.nan

    idx_f = idx[merged.loc[idx, GROUP_COL] == "F"]
    idx_m = idx[merged.loc[idx, GROUP_COL] == "M"]
    n_f, n_m = len(idx_f), len(idx_m)
    if n_f == 0 or n_m == 0:
        return np.nan

    num_f = int((pred_series.loc[idx_f] == r).sum())
    num_m = int((pred_series.loc[idx_m] == r).sum())
    p_f = num_f / n_f
    p_m = num_m / n_m
    return p_f - p_m  # may be zero; keep it

def _truth_weights_overall():
    """u_k = #(true=k)/N; NaN if N=0."""
    N = len(merged.index)
    if N == 0:
        return {1: np.nan, 2: np.nan, 3: np.nan}
    counts = {k: int((true_band == k).sum()) for k in (1,2,3)}
    return {k: counts[k] / N for k in (1,2,3)}

def _pred_weights(pred_series, sub_idx):
    idx = merged.index.intersection(pd.Index(sub_idx))
    n = len(idx)
    if n == 0:
        return {1: np.nan, 2: np.nan, 3: np.nan}
    return {r: float((pred_series.loc[idx] == r).sum()) / n for r in (1,2,3)}

def eo_table_for_model(pred_col):
    pred = merged[pred_col].astype("Int64")

    subsets = {
        "Uncond.": merged.index,                  # combine across k with truth-mix (renorm over valid k)
        "Cond@1": merged.index[true_band == 1],
        "Cond@2": merged.index[true_band == 2],
        "Cond@3": merged.index[true_band == 3],
    }

    # Per-row values
    eo_grid = {r: {} for r in (1,2,3)}

    # Cond@k columns: direct differences within S_k
    for r in (1,2,3):
        for k, col_name in [(1,"Cond@1"), (2,"Cond@2"), (3,"Cond@3")]:
            eo_grid[r][col_name] = _eo_cond_k_for_pred_r(pred, k, r)

    # Uncond. column: truth-mix weighted average over k, renormalized over valid k
    u_raw = _truth_weights_overall()
    for r in (1,2,3):
        valid_k = [k for k in (1,2,3) if not pd.isna(_eo_cond_k_for_pred_r(pred, k, r))]
        if len(valid_k) == 0:
            eo_grid[r]["Uncond."] = np.nan
            continue
        denom = sum(u_raw[k] for k in valid_k if not pd.isna(u_raw[k]))
        if denom is None or pd.isna(denom) or denom == 0:
            eo_grid[r]["Uncond."] = np.nan
            continue
        eo_grid[r]["Uncond."] = sum(u_raw[k] * _eo_cond_k_for_pred_r(pred, k, r) for k in valid_k) / denom

    # Build rows (Pred=1..3)
    rows = []
    for r in (1,2,3):
        rows.append([
            f"Pred={r}",
            eo_grid[r]["Uncond."],
            eo_grid[r]["Cond@1"],
            eo_grid[r]["Cond@2"],
            eo_grid[r]["Cond@3"],
        ])

    # Summary row (prediction-mix weights within each subset; zeros included; NaNs excluded via renorm)
    summary_vals = []
    for col_name, sub_idx in subsets.items():
        w_raw = _pred_weights(pred, sub_idx)
        valid_r = [r for r in (1,2,3) if not pd.isna(eo_grid[r][col_name])]
        if len(valid_r) == 0:
            summary_vals.append(np.nan)
            continue
        denom = sum(w_raw[r] for r in valid_r if not pd.isna(w_raw[r]))
        if denom is None or pd.isna(denom) or denom == 0:
            summary_vals.append(np.nan)
            continue
        val = sum((w_raw[r] / denom) * eo_grid[r][col_name] for r in valid_r)
        summary_vals.append(val)

    rows.append(["Summary"] + summary_vals)

    return (pd.DataFrame(rows, columns=["Row","Uncond.","Cond@1","Cond@2","Cond@3"])
              .set_index("Row"))

# Run for each method
for model_name, col in MODEL_COLS.items():
    table = eo_table_for_model(col)
    print(f"\n=== Block 2 — EO (normalized summary) — {model_name} ===")
    print(table.round(3))



=== Block 2 — EO (normalized summary) — tfidf ===
         Uncond.  Cond@1  Cond@2  Cond@3
Row                                     
Pred=1    -0.019  -0.010  -0.029   0.000
Pred=2     0.019   0.010   0.004   0.133
Pred=3    -0.000   0.000   0.025  -0.133
Summary    0.004  -0.006   0.001   0.006

=== Block 2 — EO (normalized summary) — bert ===
         Uncond.  Cond@1  Cond@2  Cond@3
Row                                     
Pred=1     0.010   0.006   0.015   0.000
Pred=2    -0.019  -0.006  -0.048   0.092
Pred=3     0.008   0.000   0.033  -0.092
Summary   -0.006   0.004  -0.039  -0.046

=== Block 2 — EO (normalized summary) — chatgpt_zeroshot ===
         Uncond.  Cond@1  Cond@2  Cond@3
Row                                     
Pred=1    -0.066  -0.061  -0.075  -0.033
Pred=2     0.066   0.061   0.075   0.033
Pred=3     0.000   0.000   0.000   0.000
Summary   -0.024  -0.054  -0.015   0.021

=== Block 2 — EO (normalized summary) — chatgpt_fewshot ===
         Uncond.  Cond@1  Cond@2  Cond

In [40]:
import numpy as np
import pandas as pd

# assumes: merged, GROUP_COL, TARGET_COL, MODEL_COLS already exist

def debug_metrics_readable(model, pred_val, true_val=None, round_to=6):
    """
    Debug one event (pred == pred_val) for a given model.
    - model: key in MODEL_COLS (e.g., "bert") or a direct column name (e.g., "y_pred_bert")
    - pred_val: 1, 2, or 3
    - true_val: None for Uncond.; 1/2/3 for Cond@k
    Returns a pandas.Series with explicit counts, probabilities, DI/DP/EO, and both weight types.
    """
    # Resolve model column
    model_col = MODEL_COLS.get(model, model)
    pred = merged[model_col].astype("Int64")
    true = merged[TARGET_COL].astype("Int64")

    # Subset (matching)
    if true_val is None:
        subset_name = "Uncond."
        idx_subset = merged.index
    else:
        k = int(true_val)
        subset_name = f"Cond@{k}"
        idx_subset = merged.index[true == k]

    sub = merged.loc[idx_subset]

    # Group membership within subset
    idx_F_subset = sub.index[sub[GROUP_COL] == "F"]
    idx_M_subset = sub.index[sub[GROUP_COL] == "M"]
    total_F_in_subset = len(idx_F_subset)
    total_M_in_subset = len(idx_M_subset)
    total_in_subset   = len(sub)

    # Event mask (pred == pred_val)
    r = int(pred_val)
    event_mask = (pred == r)

    # Event counts by group (numerators)
    event_count_F = int(event_mask.loc[idx_F_subset].sum()) if total_F_in_subset > 0 else np.nan
    event_count_M = int(event_mask.loc[idx_M_subset].sum()) if total_M_in_subset > 0 else np.nan

    # Probabilities within subset by group (denominators)
    prop_event_among_F = (event_count_F / total_F_in_subset) if total_F_in_subset > 0 else np.nan
    prop_event_among_M = (event_count_M / total_M_in_subset) if total_M_in_subset > 0 else np.nan

    # Metrics
    DI_ratio = (prop_event_among_F / prop_event_among_M) if (
        not pd.isna(prop_event_among_F) and not pd.isna(prop_event_among_M) and prop_event_among_M != 0
    ) else np.nan

    DP_diff = (prop_event_among_F - prop_event_among_M) if (
        not pd.isna(prop_event_among_F) and not pd.isna(prop_event_among_M)
    ) else np.nan

    # --- EO handling ---
    # If conditional (subset is fixed true=k), EO equals DP on that subset.
    # If unconditional, EO is truth-mix–weighted across k ∈ {1,2,3} within the SAME subset.
    ks = (1, 2, 3)
    eo_by_true_band = {}
    truth_mix_weights = {}
    if total_in_subset > 0:
        for k in ks:
            sub_k = sub.loc[sub[TARGET_COL] == k]
            n_k = len(sub_k)
            truth_mix_weights[k] = n_k / total_in_subset  # u_k(S)
            if n_k == 0:
                eo_by_true_band[k] = np.nan
                continue
            idx_F_k = sub_k.index[sub_k[GROUP_COL] == "F"]
            idx_M_k = sub_k.index[sub_k[GROUP_COL] == "M"]
            nF_k, nM_k = len(idx_F_k), len(idx_M_k)
            if nF_k == 0 or nM_k == 0:
                eo_by_true_band[k] = np.nan
                continue
            pF_k = (event_mask.loc[idx_F_k].sum()) / nF_k
            pM_k = (event_mask.loc[idx_M_k].sum()) / nM_k
            eo_by_true_band[k] = pF_k - pM_k  # zeros are valid; NaN only if group absent
    else:
        truth_mix_weights = {k: np.nan for k in ks}
        eo_by_true_band   = {k: np.nan for k in ks}

    if true_val is not None:
        EO_diff = DP_diff
    else:
        # Uncond. EO = truth-mix weighted over valid ks (renormalized)
        valid_k = [k for k in ks if not pd.isna(eo_by_true_band[k]) and not pd.isna(truth_mix_weights[k])]
        if len(valid_k) == 0:
            EO_diff = np.nan
        else:
            denom = sum(truth_mix_weights[k] for k in valid_k)
            EO_diff = sum(truth_mix_weights[k] * eo_by_true_band[k] for k in valid_k) / denom if denom else np.nan

    # --- weights you asked for ---
    # 1) prediction-mix weights within subset (w_r(S) for r=1..3)
    if total_in_subset > 0:
        prediction_mix_weights = {rr: float((pred.loc[idx_subset] == rr).sum()) / total_in_subset for rr in ks}
    else:
        prediction_mix_weights = {rr: np.nan for rr in ks}
    weight_pred_mix_selected_r = prediction_mix_weights[r]  # w_r(S)

    # Build output (rounded for readability)
    out = pd.Series({
        # identifiers
        "subset": subset_name,
        "model_col": model_col,
        "pred_val": r,
        "true_val": (None if true_val is None else int(true_val)),

        # denominators
        "total_in_subset": total_in_subset,
        "total_F_in_subset": total_F_in_subset,
        "total_M_in_subset": total_M_in_subset,

        # numerators (event counts)
        "event_count_F": event_count_F,
        "event_count_M": event_count_M,

        # probabilities
        "prop_event_among_F": prop_event_among_F,
        "prop_event_among_M": prop_event_among_M,

        # metrics
        "DI_ratio": DI_ratio,
        "DP_diff": DP_diff,
        "EO_diff": EO_diff,

        # weights (both kinds)
        "weight_pred_mix_selected_r": weight_pred_mix_selected_r,   # scalar w_r(S) for chosen r
        "prediction_mix_weights_all": prediction_mix_weights,       # {1:w1,2:w2,3:w3}
        "truth_mix_weights_all": truth_mix_weights,                 # {1:u1,2:u2,3:u3}

        # EO components (useful when Uncond.)
        "EO_by_true_band": eo_by_true_band,                         # {1:EO_1,2:EO_2,3:EO_3}
    })

    if round_to is not None:
        # Round numeric scalars; keep dicts as-is for readability
        for key in [
            "prop_event_among_F", "prop_event_among_M",
            "DI_ratio", "DP_diff", "EO_diff",
            "weight_pred_mix_selected_r"
        ]:
            if pd.notna(out[key]):
                out[key] = float(np.round(out[key], round_to))
        # Optional: round dict values too (commented to avoid altering structure)
        # out["prediction_mix_weights_all"] = {k: (None if pd.isna(v) else round(v, round_to))
        #                                      for k,v in out["prediction_mix_weights_all"].items()}
        # out["truth_mix_weights_all"]      = {k: (None if pd.isna(v) else round(v, round_to))
        #                                      for k,v in out["truth_mix_weights_all"].items()}
        # out["EO_by_true_band"]            = {k: (None if pd.isna(v) else round(v, round_to))
        #                                      for k,v in out["EO_by_true_band"].items()}

    return out


In [41]:
# Unconditional, BERT, predicted class = 2
print(debug_metrics_readable("bert", pred_val=2))

# Conditional on true band 1, same predicted class
print(debug_metrics_readable("bert", pred_val=2, true_val=1))

# Using a direct column name
print(debug_metrics_readable("y_pred_chatgpt_fewshot", pred_val=3))


subset                                                                  Uncond.
model_col                                                           y_pred_bert
pred_val                                                                      2
true_val                                                                   None
total_in_subset                                                             434
total_F_in_subset                                                           206
total_M_in_subset                                                           228
event_count_F                                                               119
event_count_M                                                               125
prop_event_among_F                                                      0.57767
prop_event_among_M                                                     0.548246
DI_ratio                                                                1.05367
DP_diff                                 